# W&B Registry — Promote the Synthetic Dataset

The **W&B Registry** is an organisation-level catalogue of
promotable artifacts. It sits on top of project-scoped artifacts:
you `log_artifact` inside a project as usual, then `link_artifact`
into a Registry **Collection** to make the artifact org-visible,
aliasable (`production`, `staging`, ...), and discoverable across
projects.

The workshop's Registry is called **`airfoil_surrogates`**. This
notebook adds a **`datasets`** Collection to it, containing the
synthetic NACA airfoil cases that both `experiment_tracking.ipynb`
and `sweeps.ipynb` consume.

**What this notebook does**
1. Generate a small synthetic NACA airfoil dataset (~80 cases).
2. Serialise it to a JSON file.
3. Log it as a project artifact, then `link_artifact` it into the
   Registry's `datasets` Collection with `production` + `v1` aliases.
4. Read it back from the Registry as a sanity check.

**Prerequisites**
- `uv sync`, `uv run wandb login`.
- Access to the `wandb_emea` entity — or edit the `ENTITY` constant
  in §1.

**Run order in the workshop**
`registry.ipynb` (this notebook) → `experiment_tracking.ipynb` →
`sweeps.ipynb` → `reports.ipynb`.


## 1. Setup

Third-party imports and section-local constants.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "physicsx_demo").exists())
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import json
import math
import tempfile

import numpy as np
import wandb

from physicsx_demo.utils.env import configure_matplotlib, load_local_env, resolve_wandb_target

configure_matplotlib()

import matplotlib.pyplot as plt

from physicsx_demo.utils.data import AirfoilCase, cases_from_payload, cases_to_json_payload, synthetic_dataset
from physicsx_demo.utils.viz import airfoil_gallery_figure, airfoil_profile_figure, airfoil_sections_figure, save_figure_png

load_local_env()
ENTITY, PROJECT = resolve_wandb_target()

REGISTRY_NAME = "airfoil_surrogates"
DATASET_COLLECTION = "datasets"
DATASET_ARTIFACT_NAME = "synthetic_airfoil_slice"

SEED = 42
N_CASES = 80
N_POINTS = 512

print(f"wandb   = {wandb.__version__}")
print(f"target  = {ENTITY}/{PROJECT}")
print(f"Registry path -> wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}")


## 2. Synthetic NACA dataset generator

Small, laptop-friendly toy dataset — a 2D neural-field CFD problem.
Each case is a NACA-4-digit airfoil under a sampled
`(angle_of_attack, Reynolds_number)`, with analytic pressure and
velocity fields evaluated at random points in the flow domain.
Singularities (`1/r²` doublet, `1/r` circulation) are clipped so the
targets stay in a physical range.

Features per point: `(x, y, sdf, aoa, log_re)`.
Targets per point: `(pressure, u, v)`.


In [ ]:
# Shared AirfoilCase, synthetic dataset generation, and airfoil plotting helpers
# live in `src/physicsx_demo/utils` so this notebook can stay focused on
# Registry logging, verification, and promotion.


## 3. Generate the dataset and serialise it

We generate the cases deterministically (via `seed`) and write them
to a JSON file. The schema is self-describing so any downstream
notebook — Python or otherwise — can load it without extra tooling.


In [ ]:
cases = synthetic_dataset(N_CASES, n_points=N_POINTS, seed=SEED)

sample = cases[0]
print(f"total cases: {len(cases)}    per-case: N={N_POINTS}, features={sample.features().shape[-1]}, targets={sample.targets().shape[-1]}")
print(f"first case: aoa={math.degrees(sample.aoa):.2f} deg,  Re={sample.re:.1f}")

tmp_dir = Path(tempfile.mkdtemp())
dataset_path = tmp_dir / f"{DATASET_ARTIFACT_NAME}.json"
media_dir = tmp_dir / "dataset_media"
media_dir.mkdir(parents=True, exist_ok=True)

payload = cases_to_json_payload(cases, seed=SEED, n_points=N_POINTS)
with open(dataset_path, "w") as f:
    json.dump(payload, f)

print(f"wrote JSON: {dataset_path}   ({dataset_path.stat().st_size / 1024:.1f} KB)")

profile_columns = ["case_id", "aoa_deg", "re", "airfoil_profile", "profile_sections"]
profile_table = wandb.Table(columns=profile_columns)
profile_artifact_table = wandb.Table(columns=profile_columns)
for i, c in enumerate(cases[:12]):
    fig_profile = airfoil_profile_figure(c, title=f"dataset case {i} profile")
    profile_path = save_figure_png(fig_profile, media_dir / "airfoil_profiles", f"case_{i:02d}_airfoil_profile")
    plt.close(fig_profile)

    fig_sections = airfoil_sections_figure(c, title=f"dataset case {i} chord sections")
    sections_path = save_figure_png(fig_sections, media_dir / "airfoil_sections", f"case_{i:02d}_airfoil_sections")
    plt.close(fig_sections)

    scalar_row = [i, float(math.degrees(c.aoa)), float(c.re)]
    profile_table.add_data(
        *scalar_row,
        wandb.Image(str(profile_path), caption=f"case {i} airfoil profile"),
        wandb.Image(str(sections_path), caption=f"case {i} chord sections"),
    )
    profile_artifact_table.add_data(
        *scalar_row,
        wandb.Image(str(profile_path), caption=f"case {i} airfoil profile"),
        wandb.Image(str(sections_path), caption=f"case {i} chord sections"),
    )

fig_gallery = airfoil_gallery_figure(cases, "Synthetic dataset airfoil profile gallery")
gallery_path = save_figure_png(fig_gallery, media_dir / "airfoil_profiles", "dataset_airfoil_profile_gallery")
plt.close(fig_gallery)

print("prepared dataset profile/section table with 12 preview airfoils")


## 4. Log + promote to the Registry

The two-step pattern:

1. `run.log_artifact(...)` — creates a **project-level** artifact
   (versioned, but only visible inside this project).
2. `run.link_artifact(target_path="wandb-registry-{R}/{C}", aliases=[...])`
   — promotes that specific version into the Registry Collection.
   Org-visible, aliased, cross-project consumable. The Collection is
   created implicitly on first link; the Registry itself is created
   via `api.create_registry(...)` inside a `try/except` since
   `experiment_tracking.ipynb` may have created it already.

`log_artifact` is asynchronous — we call `.wait()` on the returned
reference before linking so the alias resolves on the backend.


In [ ]:
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="dataset-registration",
    tags=["notebook", "registry", "dataset"],
    name="dataset-register-and-promote",
) as run:
    
    artifact = wandb.Artifact(
        name=DATASET_ARTIFACT_NAME,
        type="dataset",
        description="Synthetic NACA-4-digit airfoil cases with hidden global wake/circulation terms (workshop dataset).",
        metadata={"n_cases": N_CASES, "n_points": N_POINTS, "seed": SEED, "generator_version": "v2_global_wake"},
    )
    artifact.add_file(str(dataset_path))
    artifact.add(profile_artifact_table, "airfoil_shape_sections")
    artifact.add_file(str(gallery_path), name="airfoil_profile_gallery.png")
    run.log({
        "dataset/airfoil_profile_gallery": wandb.Image(str(gallery_path), caption="Dataset airfoil profile gallery"),
        "dataset/airfoil_shape_sections_table": profile_table,
    })
    logged = run.log_artifact(artifact)
    logged.wait()                                 # ensure upload commits



    api = wandb.Api(overrides={"entity": ENTITY, "project": PROJECT})
    try:
        registry = api.create_registry(
            name=REGISTRY_NAME,
            visibility="organization",
        )
        print(f"created registry: {registry.name}")
    except Exception as exc:
        print(f"registry create skipped (may already exist): {exc}")

    target_path = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}"
    run.link_artifact(
        logged,
        target_path=target_path,
        aliases=["production", "v1"],
    )
    
    print(f"Dataset: {DATASET_ARTIFACT_NAME}")
    print(f"linked -> {target_path}   aliases: production, v1")
    print(f"run url: {run.url}")


## 5. Verify — read the dataset back from the Registry

Sanity-check: a fresh downstream run consumes the promoted entry.
This is exactly what `experiment_tracking.ipynb` and `sweeps.ipynb`
do at their setup step.


In [ ]:
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="registry-verify",
    tags=["notebook", "registry", "verify"],
    name="dataset-verify",
) as run:
    art = run.use_artifact(f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production")
    download_dir = Path(art.download())
    json_file = next(download_dir.glob("*.json"))
    with open(json_file) as f:
        loaded = json.load(f)

    print(f"pulled {json_file.name}   "
          f"({json_file.stat().st_size / 1024:.1f} KB, "
          f"{len(loaded['cases'])} cases)")
    print(f"meta: {loaded['meta']}")

    verify_cases = cases_from_payload(loaded, limit=6)
    verify_columns = ["case_id", "aoa_deg", "re", "airfoil_profile", "profile_sections"]
    verify_profile_table = wandb.Table(columns=verify_columns)
    verify_artifact_table = wandb.Table(columns=verify_columns)
    verify_media_dir = Path(tempfile.mkdtemp()) / "verified_dataset_media"
    verify_media_dir.mkdir(parents=True, exist_ok=True)
    for i, c in enumerate(verify_cases):
        fig_profile = airfoil_profile_figure(c, title=f"verified case {i} profile")
        profile_path = save_figure_png(fig_profile, verify_media_dir / "airfoil_profiles", f"verified_case_{i:02d}_airfoil_profile")
        plt.close(fig_profile)

        fig_sections = airfoil_sections_figure(c, title=f"verified case {i} chord sections")
        sections_path = save_figure_png(fig_sections, verify_media_dir / "airfoil_sections", f"verified_case_{i:02d}_airfoil_sections")
        plt.close(fig_sections)

        scalar_row = [i, float(math.degrees(c.aoa)), float(c.re)]
        verify_profile_table.add_data(
            *scalar_row,
            wandb.Image(str(profile_path), caption=f"verified case {i} airfoil profile"),
            wandb.Image(str(sections_path), caption=f"verified case {i} chord sections"),
        )
        verify_artifact_table.add_data(
            *scalar_row,
            wandb.Image(str(profile_path), caption=f"verified case {i} airfoil profile"),
            wandb.Image(str(sections_path), caption=f"verified case {i} chord sections"),
        )

    verify_media_artifact = wandb.Artifact(
        "verified_dataset_airfoil_sections",
        type="media-table",
        description="Verified Registry dataset airfoil profile and chord-section images.",
    )
    verify_media_artifact.add(verify_artifact_table, "verified_airfoil_shape_sections")
    logged_verify_media = run.log_artifact(verify_media_artifact, aliases=["latest"])
    run.log({"dataset/verified_airfoil_shape_sections_table": verify_profile_table})
    run.summary["dataset/verified_airfoil_sections_artifact"] = "verified_dataset_airfoil_sections:latest"
    logged_verify_media.wait()


## 6. Wrap up — what to look at in the W&B UI

**Runs** (two)
- `dataset-register-and-promote` (job_type = `dataset-registration`)
- `dataset-verify` (job_type = `registry-verify`)

**Registry**
- `wandb-registry-airfoil_surrogates/datasets` now contains one
  version of `synthetic_airfoil_slice`, aliased `production` + `v1`.

**Lineage graph**
- `synthetic_airfoil_slice:v0` produced by the register-run and
  consumed by the verify-run.

**Downstream** — `experiment_tracking.ipynb` and `sweeps.ipynb` both
resolve the dataset at
`wandb-registry-airfoil_surrogates/datasets:production` at their
setup step. Run those notebooks next.
